# Practica 1 (30 min): GCN vs GAT vs GraphSAGE
## Master Oficial: Big Data Science

### Objetivo
Comparar 3 arquitecturas GNN bajo el mismo protocolo y reportar metricas de rendimiento.

### Entregables
1. Tabla comparativa con `test_acc` y `time_sec` para GCN, GAT y GraphSAGE.
2. Mini ablation cambiando `hidden_dim` en un modelo.
3. Interpretacion breve (2-3 lineas).

### Dinamica sugerida (equipos de 3)
- Persona A: completa TODO 1 (`build_model`).
- Persona B: completa TODO 2 (`train_one_run`).
- Persona C: completa TODO 3 (`run_experiment`) y tabla final.

### Tiempo
30 minutos.

### Checkpoints
- Checkpoint 1: `build_model` produce logits con shape correcto.
- Checkpoint 2: `train_one_run` devuelve metricas validas.
- Checkpoint 3: `run_experiment` genera tabla para 3 modelos.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATv2Conv, SAGEConv

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device} | Data dir: {DATA_DIR.resolve()}")

In [ ]:
def load_dataset_with_fallback(data_dir: Path):
    last_error = None
    for name in ["Cora", "CiteSeer"]:
        try:
            ds = Planetoid(root=str(data_dir / "planetoid"), name=name)
            return ds, ds[0], name
        except Exception as exc:
            last_error = exc
            print(f"Aviso: no se pudo cargar {name}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"No se pudo cargar Cora ni CiteSeer: {last_error}")

dataset, data, dataset_name = load_dataset_with_fallback(DATA_DIR)
data = data.to(device)

print(f"Dataset activo: {dataset_name}")
print(
    f"nodes={data.num_nodes} | edges={data.num_edges} | features={dataset.num_features} | classes={dataset.num_classes}"
)
print(
    f"train={int(data.train_mask.sum())} | val={int(data.val_mask.sum())} | test={int(data.test_mask.sum())}"
)

assert data.x.shape[0] == data.y.shape[0], "Mismatch entre nodos y etiquetas"
assert int(data.train_mask.sum()) > 0, "train_mask vacia"
print("Checkpoint loader OK")

In [ ]:
class GCNNet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.conv1(h, edge_index)
        h = torch.relu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.conv2(h, edge_index)
        return logits, emb


class GATNet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, out_dim, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.gat2(h, edge_index)
        return logits, emb


class SAGENet(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.s1 = SAGEConv(in_dim, hidden_dim)
        self.s2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        h = self.s1(x, edge_index)
        h = F.relu(h)
        emb = h
        h = F.dropout(h, p=self.dropout, training=self.training)
        logits = self.s2(h, edge_index)
        return logits, emb

In [ ]:
# TODO 1
# Completa esta funcion para devolver el modelo correcto segun model_name.
# model_name permitido: 'gcn', 'gat', 'sage'

def build_model(model_name, in_dim, out_dim, device, hidden_dim=None, heads=4):
    model_name = model_name.lower()

    if hidden_dim is None:
        hidden_dim = 16 if model_name == "gat" else 64

    # TODO: reemplaza este bloque por la logica correcta
    # if model_name == "gcn":
    #     model = ...
    # elif model_name == "gat":
    #     model = ...
    # elif model_name == "sage":
    #     model = ...
    # else:
    #     raise ValueError(...)

    _ = (in_dim, out_dim, device, heads)
    raise NotImplementedError("Completa TODO 1: build_model")


# Checkpoint 1 (se activa cuando completes TODO 1)
try:
    m = build_model("gcn", dataset.num_features, dataset.num_classes, device)
    m.eval()
    with torch.no_grad():
        logits, emb = m(data.x, data.edge_index)
    assert logits.shape == (data.num_nodes, dataset.num_classes), "Shape de logits incorrecto"
    assert emb.shape[0] == data.num_nodes, "Embeddings con numero de nodos incorrecto"
    print("Checkpoint 1 OK")
except NotImplementedError:
    print("Completa TODO 1 para activar Checkpoint 1")

In [ ]:
def accuracy(logits, y):
    pred = logits.argmax(dim=1)
    return float((pred == y).sum().item() / len(y))


def train_one_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())


@torch.no_grad()
def evaluate(model, data, criterion):
    model.eval()
    logits, _ = model(data.x, data.edge_index)

    val_loss = float(criterion(logits[data.val_mask], data.y[data.val_mask]).item())
    val_acc = accuracy(logits[data.val_mask], data.y[data.val_mask])
    test_acc = accuracy(logits[data.test_mask], data.y[data.test_mask])

    return val_loss, val_acc, test_acc

In [ ]:
# TODO 2
# Completa esta funcion para entrenar un modelo y devolver metricas finales.

def train_one_run(model_name, seed=42, hidden_dim=None, max_epochs=80, patience=15):
    # TODO: fija seed
    # seed_everything(seed)

    # TODO: construye el modelo usando build_model
    # model = ...

    # TODO: define optimizer y criterion
    # optimizer = ...
    # criterion = ...

    # TODO: implementa ciclo de entrenamiento con early stopping simple por val_loss
    # pista: guarda best_state y corta cuando wait >= patience

    # TODO: al final calcula val_acc y test_acc y retorna dict con metadatos
    # return {
    #   'model': model_name,
    #   'seed': seed,
    #   'hidden_dim': hidden_dim if hidden_dim is not None else (16 if model_name == 'gat' else 64),
    #   'val_acc': ...,
    #   'test_acc': ...,
    #   'time_sec': ...
    # }

    _ = (model_name, seed, hidden_dim, max_epochs, patience)
    raise NotImplementedError("Completa TODO 2: train_one_run")


# Checkpoint 2 (se activa cuando completes TODO 2)
try:
    test_row = train_one_run("gcn", seed=42, max_epochs=20, patience=5)
    assert 0.0 <= test_row["test_acc"] <= 1.0, "test_acc fuera de rango"
    assert test_row["time_sec"] >= 0.0, "time_sec invalido"
    print("Checkpoint 2 OK")
    print(test_row)
except NotImplementedError:
    print("Completa TODO 2 para activar Checkpoint 2")

In [ ]:
# TODO 3
# Completa run_experiment para correr gcn/gat/sage y devolver un DataFrame ordenado por test_acc.

def run_experiment(seed=42):
    # TODO: itera sobre ['gcn', 'gat', 'sage'] y llama train_one_run
    # rows = []
    # ...
    # df = pd.DataFrame(rows).sort_values(by='test_acc', ascending=False).reset_index(drop=True)
    # return df

    _ = seed
    raise NotImplementedError("Completa TODO 3: run_experiment")


try:
    results_df = run_experiment(seed=42)
    assert len(results_df) == 3, "La tabla debe tener 3 filas (gcn/gat/sage)"
    print("Checkpoint 3 OK")
    display(results_df)
except NotImplementedError:
    print("Completa TODO 3 para activar Checkpoint 3")

In [ ]:
# Ablation guiada (sin TODO adicional)
# Cambia solo estas dos lineas y vuelve a ejecutar.
ABLATION_MODEL = "gat"
HIDDEN_DIMS = [8, 16, 32]

ablation_rows = []
for h in HIDDEN_DIMS:
    try:
        row = train_one_run(ABLATION_MODEL, seed=42, hidden_dim=h, max_epochs=60, patience=10)
        ablation_rows.append(row)
    except NotImplementedError:
        print("Completa TODO 2 antes de ejecutar el ablation")
        break

if len(ablation_rows) > 0:
    ablation_df = pd.DataFrame(ablation_rows).sort_values(by="hidden_dim").reset_index(drop=True)
    display(ablation_df)

    plt.figure(figsize=(6, 4))
    plt.plot(ablation_df["hidden_dim"], ablation_df["test_acc"], marker="o")
    plt.title(f"Ablation: {ABLATION_MODEL.upper()} hidden_dim vs test_acc")
    plt.xlabel("hidden_dim")
    plt.ylabel("test_acc")
    plt.grid(alpha=0.3)
    plt.show()

## Respuesta corta (2-3 lineas)

1. Que modelo rindio mejor y por que crees?
2. Que trade-off observas entre accuracy y tiempo?
3. Que cambio con el ablation y como lo interpretas?